In [2]:
import openmeteo_requests
import pandas as pd
import numpy as np
from retry_requests import retry
import requests_cache

In [3]:
cache_session = requests_cache.CachedSession('.cache', expire_after=3600)
retry_session = retry(cache_session, retries=5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session=retry_session)

In [5]:
url = "https://historical-forecast-api.open-meteo.com/v1/forecast"
params = {
    "latitude": 47.1221,
	"longitude": 9.486,
	"start_date": "2024-01-01",
	"end_date": "2024-01-31",
	"daily": ["temperature_2m_max", "temperature_2m_min", "precipitation_sum", "temperature_2m_mean"],
	"timezone": "Europe/Berlin",
}

In [6]:
responses = openmeteo.weather_api(url=url, params=params)

In [10]:
response = responses[0]

In [15]:
print("Location: Sevelen SG, Switzerland")
print(f"Coordinates: {response.Latitude():.2f}°N {response.Longitude():.2f}°E")
print(f"Elevation: {response.Elevation()} m asl")

Location: Sevelen SG, Switzerland
Coordinates: 47.12°N 9.48°E
Elevation: 462.0 m asl


In [16]:
daily = response.Daily()

In [27]:
daily_temperature_2m_max = daily.Variables(0).ValuesAsNumpy()
daily_temperature_2m_min = daily.Variables(1).ValuesAsNumpy()
daily_precipitation_sum = daily.Variables(2).ValuesAsNumpy()
daily_temperature_2m_mean = daily.Variables(3).ValuesAsNumpy()

In [36]:
daily_data = {
    "date": pd.date_range(
        start = pd.to_datetime(daily.Time(), unit = "s", utc=True),
        end = pd.to_datetime(daily.TimeEnd(), unit = "s", utc=True),
        freq = pd.Timedelta(seconds = daily.Interval()),
        inclusive="left"
    ).tz_convert(response.Timezone().decode())
}

In [37]:
daily_data["temperature_2m_max"] = daily_temperature_2m_max
daily_data["temperature_2m_min"] = daily_temperature_2m_min
daily_data["precipitation_sum"] = daily_precipitation_sum
daily_data["temperature_2m_mean"] = daily_temperature_2m_mean

In [38]:
daily_dataframe = pd.DataFrame(data=daily_data)

In [39]:
daily_dataframe

,date,temperature_2m_max,temperature_2m_min,precipitation_sum,temperature_2m_mean
0,2023-12-31 23:00:00+01:00,7.476,1.076,0.500000,3.928333
1,2024-01-01 23:00:00+01:00,11.076,2.326,0.900000,7.513500
2,2024-01-02 23:00:00+01:00,13.926,6.776,7.700000,9.480167
3,2024-01-03 23:00:00+01:00,9.126,1.576,9.700001,6.586417
4,2024-01-04 23:00:00+01:00,7.376,3.176,1.700000,5.240583
5,2024-01-05 23:00:00+01:00,4.326,2.126,12.599999,3.203083
6,2024-01-06 23:00:00+01:00,2.476,-0.374,13.700000,1.353083
7,2024-01-07 23:00:00+01:00,-0.224,-2.774,1.300000,-1.680250
8,2024-01-08 23:00:00+01:00,-1.424,-3.974,0.000000,-2.563583
9,2024-01-09 23:00:00+01:00,0.276,-4.474,0.000000,-2.638583
